# Big Data Analysis using PySpark

Dataset: Netflix Movies and TV Shows

Objective: Analyze large dataset using PySpark to demonstrate scalability.

In [1]:
pip install pyspark

In [3]:
### Loading Libraries

In [10]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
import pandas as pd

In [5]:
spark = SparkSession.builder \
    .appName("Big Data Analysis") \
    .getOrCreate()

In [11]:
df = spark.read.csv("netflix_titles.csv", header=True, inferSchema=True)

df.show(5)

+-------+-------+--------------------+---------------+--------------------+-------------+------------------+------------+------+---------+--------------------+--------------------+
|show_id|   type|               title|       director|                cast|      country|        date_added|release_year|rating| duration|           listed_in|         description|
+-------+-------+--------------------+---------------+--------------------+-------------+------------------+------------+------+---------+--------------------+--------------------+
|     s1|  Movie|Dick Johnson Is Dead|Kirsten Johnson|                NULL|United States|September 25, 2021|        2020| PG-13|   90 min|       Documentaries|As her father nea...|
|     s2|TV Show|       Blood & Water|           NULL|Ama Qamata, Khosi...| South Africa|September 24, 2021|        2021| TV-MA|2 Seasons|International TV ...|After crossing pa...|
|     s3|TV Show|           Ganglands|Julien Leclercq|Sami Bouajila, Tr...|         NULL|Septem

In [19]:
#### checking dataset structure

In [18]:
df.printSchema()

root
 |-- show_id: string (nullable = true)
 |-- type: string (nullable = true)
 |-- title: string (nullable = true)
 |-- director: string (nullable = true)
 |-- cast: string (nullable = true)
 |-- country: string (nullable = true)
 |-- date_added: string (nullable = true)
 |-- release_year: string (nullable = true)
 |-- rating: string (nullable = true)
 |-- duration: string (nullable = true)
 |-- listed_in: string (nullable = true)
 |-- description: string (nullable = true)



In [20]:
#### counting total records

In [21]:
df.count()

8809

In [ ]:
#### Checking Missing Values

In [22]:
df.select([count(when(col(c).isNull(), c)).alias(c) for c in df.columns]).show()

+-------+----+-----+--------+----+-------+----------+------------+------+--------+---------+-----------+
|show_id|type|title|director|cast|country|date_added|release_year|rating|duration|listed_in|description|
+-------+----+-----+--------+----+-------+----------+------------+------+--------+---------+-----------+
|      0|   1|    2|    2636| 826|    832|        13|           2|     6|       5|        3|          3|
+-------+----+-----+--------+----+-------+----------+------------+------+--------+---------+-----------+



In [23]:
#### Data Cleaning

In [24]:
df_clean = df.dropna(subset=["country","type","release_year"])

In [25]:
#### Removing Duplicate Records

In [26]:
df_clean = df_clean.dropDuplicates()

In [28]:
#### Converting Date Column

In [27]:
from pyspark.sql.functions import to_date

df_clean = df_clean.withColumn(
    "date_added",
    to_date(col("date_added"), "MMMM d, yyyy")
)

In [ ]:
#### Verify Clean Data

In [29]:
df_clean.show(5)

+-------+-----+--------------------+--------------------+--------------------+--------------------+----------+------------+------+--------+--------------------+--------------------+
|show_id| type|               title|            director|                cast|             country|date_added|release_year|rating|duration|           listed_in|         description|
+-------+-----+--------------------+--------------------+--------------------+--------------------+----------+------------+------+--------+--------------------+--------------------+
|   s461|Movie|           Surf's Up|Ash Brannon, Chri...|Shia LaBeouf, Jef...|United States, Ca...|2021-07-15|        2007|    PG|  86 min|Children & Family...|This Oscar-nomina...|
|   s695|Movie|               Aziza|      Soudade Kaadan|Caress Bashar, Ab...|      Lebanon, Syria|2021-06-17|        2019| TV-PG|  13 min|Comedies, Dramas,...|This short film f...|
|   s883|Movie|Jungle Beat: The ...|         Brent Dawes|Ed Kear, David Me...|           M

In [30]:
#### Movies vs TV Shows Analysis

In [31]:
df_clean.groupBy("type").count().show()

+-------------+-----+
|         type|count|
+-------------+-----+
|      TV Show| 2285|
|        Movie| 5691|
|William Wyler|    1|
+-------------+-----+



In [32]:
#### Top 10 Countries Producing Content

In [33]:
df_clean.groupBy("country") \
.count() \
.orderBy("count", ascending=False) \
.show(10)

+--------------+-----+
|       country|count|
+--------------+-----+
| United States| 2805|
|         India|  972|
|United Kingdom|  419|
|         Japan|  245|
|   South Korea|  199|
|        Canada|  181|
|         Spain|  145|
|        France|  123|
|        Mexico|  110|
|         Egypt|  106|
+--------------+-----+
only showing top 10 rows


In [34]:
#### Content Released Per Year

In [35]:
df_clean.groupBy("release_year") \
.count() \
.orderBy("release_year") \
.show()

+-----------------+-----+
|     release_year|count|
+-----------------+-----+
|   Charles Rocket|    1|
|          Dr. Dre|    1|
|   Francis Weddey|    1|
|     Imanol Arias|    1|
|      Jade Eshete|    1|
| Kristen Johnston|    1|
| Marquell Manning|    1|
|       Nick Kroll|    1|
|    Nse Ikpe-Etim|    1|
|       Paul Sambo|    1|
|   Peter Ferriero|    1|
|     Ted Ferguson|    1|
| Álvaro Cervantes|    1|
|             1942|    2|
|             1943|    3|
|             1944|    2|
|             1945|    4|
|             1946|    2|
|             1947|    1|
|             1954|    2|
+-----------------+-----+
only showing top 20 rows


In [36]:
#### Most Common Ratings

In [37]:
df_clean.groupBy("rating") \
.count() \
.orderBy("count", ascending=False) \
.show()

+----------------+-----+
|          rating|count|
+----------------+-----+
|           TV-MA| 2920|
|           TV-14| 1928|
|               R|  785|
|           TV-PG|  772|
|           PG-13|  481|
|              PG|  280|
|           TV-Y7|  236|
|            TV-Y|  227|
|            TV-G|  190|
|              NR|   80|
|               G|   41|
|        TV-Y7-FV|    5|
|              UR|    3|
|            NULL|    3|
|           NC-17|    3|
|            2021|    2|
|   Adriane Lenox|    1|
|            2019|    1|
|     Jide Kosoko|    1|
| Jowharah Jones"|    1|
+----------------+-----+
only showing top 20 rows


In [38]:
#### Most Popular Genres

In [39]:
df_clean.groupBy("listed_in") \
.count() \
.orderBy("count", ascending=False) \
.show(10)

+--------------------+-----+
|           listed_in|count|
+--------------------+-----+
|       Documentaries|  342|
|Dramas, Internati...|  336|
|     Stand-Up Comedy|  303|
|Comedies, Dramas,...|  259|
|Dramas, Independe...|  243|
|Children & Family...|  181|
|            Kids' TV|  176|
|Documentaries, In...|  165|
|Dramas, Internati...|  163|
|Comedies, Interna...|  155|
+--------------------+-----+
only showing top 10 rows


**Insights**


Movies are more common than TV Shows on Netflix.
The United States produces the highest number of titles.
Netflix content increased significantly after 2015.
Drama and International Movies are among the most common genres.
PySpark efficiently processes large datasets for analysis.

**Conclusion**

In this task, PySpark was used to analyze a large dataset of Netflix titles.
Data preprocessing was performed to handle missing values and duplicates.
Several analyses were conducted to understand content distribution, production countries, and release trends.

This demonstrates the scalability and efficiency of PySpark for big data processing.